# Multilingual Health Q&A  Low-Resource African Languages
### Zindi Competition Notebook (VS Code / Local)

**Task:** Given a health question in Amharic, Luganda, Akan, Swahili, or English  generate a fluent, accurate answer in the **same language**.

| Baseline | Approach | When to submit |
|---|---|---|
| **Baseline 1** | TF-IDF retrieval | First submission  anchor score |
| **Baseline 2** | Zero-shot LLM (mT5 / NLLB) | Second submission |
| **Baseline 3** | Fine-tuned LLM | All subsequent experiments |

**Evaluation:** ROUGE-1 F1 (×0.37) + ROUGE-L F1 (×0.37) + LLM-as-a-Judge (×0.26)



## 1 — Install Dependencies

In [40]:
#once restart kernel after if needed
import subprocess, sys
pkgs = [
    "scikit-learn", "pandas", "numpy", "rouge-score",
    "transformers", "sentencepiece", "accelerate", "datasets",
    "torch", "matplotlib"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Packages installed")

Packages installed


## 2. Imports & Seed

In [41]:
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Define DEVICE globally so all sections can use it
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

print(f"Imports complete | Seed: {SEED} | Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Imports complete | Seed: 42 | Device: cpu


## 3. Set File Paths

Upload the competition zip to your Google Drive under `MyDrive/zindi_health_qa/` first, then run this cell.

In [42]:
from pathlib import Path

DATA_DIR       = Path("./zindi_health_qa_data")
OUTPUT_DIR     = Path("./outputs")
CHECKPOINT_DIR = Path("./checkpoints")

OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

TRAIN_PATH      = DATA_DIR / "Train.csv"
VAL_PATH        = DATA_DIR / "Val.csv"
TEST_PATH       = DATA_DIR / "Test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"

OUTPUT_TFIDF     = OUTPUT_DIR / "submission_tfidf_baseline.csv"
OUTPUT_LLM       = OUTPUT_DIR / "submission_llm_zeroshot.csv"
OUTPUT_FINETUNED = OUTPUT_DIR / "submission_finetuned.csv"

print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Output dir : {OUTPUT_DIR.resolve()}")
print()
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    status = "✅" if p.exists() else "❌  NOT FOUND"
    print(f"  {status}  {p.name}")

Data dir   : C:\Users\HP\Downloads\summative multi langiage\zindi_health_qa_data
Output dir : C:\Users\HP\Downloads\summative multi langiage\outputs

  ✅  Train.csv
  ✅  Val.csv
  ✅  Test.csv
  ✅  SampleSubmission.csv


## 4 — Load & Explore the Data

In [43]:
train  = pd.read_csv(TRAIN_PATH)
val    = pd.read_csv(VAL_PATH)
test   = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

print(f"Train  shape : {train.shape}")
print(f"Val    shape : {val.shape}")
print(f"Test   shape : {test.shape}")
print(f"Sample shape : {sample.shape}")
print()
print("Train  columns:", train.columns.tolist())
print("Test   columns:", test.columns.tolist())
print("Sample columns:", sample.columns.tolist())
train.head(3)

Train  shape : (29815, 4)
Val    shape : (6686, 4)
Test   shape : (2618, 3)
Sample shape : (2618, 4)

Train  columns: ['ID', 'input', 'output', 'subset']
Test   columns: ['ID', 'input', 'subset']
Sample columns: ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']


,ID,input,output,subset
0,ID_TR_Aka_Gha_A3B1799D,Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye w...,Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na w...,Aka_Gha
1,ID_TR_Aka_Gha_1C80317F,"Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu taa de di dwuma, na yɛbɛyɛ dɛn ahwɛ ahu sɛ yɛde redi dwuma yiye?","Wɔ Ghana mu no, amanmmra no gye binary gender nkutoo tom a she/he edinnsiananmu nkutoo na ɛka ho",Aka_Gha
2,ID_TR_Aka_Gha_06671AD1,Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔn ho fi ho anaa nna mu adwumadi a wɔtwentwɛn so no boa ma asiane so tew?,"Sɛ wɔtwe wɔn ho fi nna mu anaasɛ wɔtwentwɛn wɔn nan ase a, ɛboa ma asiane nso tew denam asiane a ɛwɔ STI ne HIV a ɛb...",Aka_Gha


In [44]:
# Column name constants (match the actual CSV headers)
ID_COL       = "ID"
QUESTION_COL = "input"
ANSWER_COL   = "output"
LANG_COL     = "subset"    # e.g. "Amh_Eth", "Lug_Uga", "Aka_Gha", "Swa_Ken", "Eng_Gha"

# language code mapping
SUBSET_TO_LANGUAGE = {
    "Eng": "English",
    "Aka": "Akan",
    "Lug": "Luganda",
    "Swa": "Swahili",
    "Amh": "Amharic",
}

def subset_to_language_name(subset_code: str) -> str:
    if not subset_code or not isinstance(subset_code, str):
        return "English"
    prefix = subset_code.split("_")[0]
    return SUBSET_TO_LANGUAGE.get(prefix, subset_code)

print("Language distribution in TRAIN:")
print(train[LANG_COL].value_counts())
print()
print("Language distribution in VAL:")
print(val[LANG_COL].value_counts())
print()
print("Answer length stats (words):")
train["_answer_len"] = train[ANSWER_COL].str.split().str.len()
print(train["_answer_len"].describe().round(1))
train.drop(columns=["_answer_len"], inplace=True)

Language distribution in TRAIN:
subset
Eng_Uga    7624
Aka_Gha    4455
Eng_Gha    4443
Eng_Eth    3915
Lug_Uga    3383
Eng_Ken    2080
Swa_Ken    2070
Amh_Eth    1845
Name: count, dtype: int64

Language distribution in VAL:
subset
Eng_Uga    1688
Aka_Gha    1114
Eng_Gha    1104
Lug_Uga     846
Eng_Eth     564
Swa_Ken     518
Amh_Eth     462
Eng_Ken     390
Name: count, dtype: int64

Answer length stats (words):
count    29815.0
mean        76.2
std         58.9
min          1.0
25%         30.0
50%         61.0
75%        107.0
max        482.0
Name: _answer_len, dtype: float64


In [45]:
# Quick look: one example per language
for lang in sorted(train[LANG_COL].unique()):
    row = train[train[LANG_COL] == lang].iloc[0]
    print(f"[{lang}]  ({subset_to_language_name(lang)})")
    print(f"  Q: {row[QUESTION_COL][:120]}")
    print(f"  A: {row[ANSWER_COL][:200]}")
    print()

[Aka_Gha]  (Akan)
  Q: Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye wɔn w
  A: Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na wɔagye wɔn nkate atom a wɔremmu atɛn anaasɛ wɔmfa asodi nto wɔn so. Wɔn a wɔbɛhyɛ wɔn

[Amh_Eth]  (Amharic)
  Q: የረጅም ጊዜ ጤናን በተመለከተ በጣም ሊያሳስበኝ የሚገቡ ሶስት ዋና ዋና ኢንፌክሽኖች የትኞቹ ናቸው?
  A: ኤች አይ ቪ፣ ሄፓታይተስ ቢ እና ሂውማን ፓፒሎማ ቫይረስ (HPV)ናቸው።

[Eng_Eth]  (English)
  Q: How is syphilis diagnosed?
  A: Blood tests like VDRL and RPR confirm syphilis infection.

[Eng_Gha]  (English)
  Q: What should I do if I have unprotected sex or experience a contraceptive failure?
  A: If you have unprotected sex or experience contraceptive failure, consider the following steps: - Take emergency contraception if appropriate and available, following the instructions provided. - Seek 

[Eng_Ken]  (English)
  Q: How can society ensure that HIV patients are incor

## 5 — Text Cleaning

In [46]:
def clean_text(x):
    """Strip whitespace, collapse spaces, handle nulls. Preserves all scripts."""
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

# Train and val — safe to drop empty rows
for df in [train, val]:
    df[QUESTION_COL] = df[QUESTION_COL].map(clean_text)
    df[ANSWER_COL]   = df[ANSWER_COL].map(clean_text)

train = train[(train[QUESTION_COL] != "") & (train[ANSWER_COL] != "")].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != "") & (val[ANSWER_COL] != "")].reset_index(drop=True)

# Test — NEVER drop rows, replace empty with placeholder
test[QUESTION_COL] = test[QUESTION_COL].map(clean_text)
test[QUESTION_COL] = test[QUESTION_COL].replace("", "what is the health advice")

# Verify
sample_check = pd.read_csv(SAMPLE_SUB_PATH)
assert len(test) == len(sample_check), \
    f"❌ Row mismatch: test={len(test)} vs sample={len(sample_check)}"

print(f"Cleaned — Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}")
print(f"✅ Test rows match sample submission ({len(sample_check):,})")

Cleaned — Train: 29,814  Val: 6,686  Test: 2,618
✅ Test rows match sample submission (2,618)


## 6 — Evaluation Utilities (ROUGE)

Whitespace tokenisation  safe for Amharic (Ge'ez script) and all other scripts.

In [47]:
from rouge_score import rouge_scorer

class WhitespaceTokenizer:
    """Language-agnostic whitespace tokeniser."""
    def tokenize(self, text):
        return str(text).strip().split() if text else []

_scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    tokenizer=WhitespaceTokenizer(),
    use_stemmer=False,
)

def compute_rouge(predictions, references):
    r1, rl = [], []
    for pred, ref in zip(predictions, references):
        s = _scorer.score(str(ref), str(pred))
        r1.append(s["rouge1"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
    return {
        "rouge1_f1": float(np.mean(r1)) if r1 else 0.0,
        "rougeL_f1": float(np.mean(rl)) if rl else 0.0,
    }

def compute_rouge_by_language(predictions, references, languages):
    rows = {}
    for lang in sorted(set(languages)):
        mask = [l == lang for l in languages]
        p = [p for p, m in zip(predictions, mask) if m]
        r = [r for r, m in zip(references, mask) if m]
        rows[lang] = compute_rouge(p, r)
    return pd.DataFrame(rows).T

print(" ROUGE scorer ready")

 ROUGE scorer ready


## 7 — Submission Builder

Run this cell now — `make_submission()` is used by all three baselines below.

In [48]:
def make_submission(ids, predictions, output_path, label=""):
    """
    Build and save a valid Zindi submission CSV.
    Columns required: ID, TargetRLF1, TargetR1F1, TargetLLM  (all three targets identical).
    """
    clean_preds = [re.sub(r"<extra_id_\d+>", "", str(p)).strip() for p in predictions]

    sub = pd.DataFrame({
        "ID":         ids,
        "TargetRLF1": clean_preds,
        "TargetR1F1": clean_preds,
        "TargetLLM":  clean_preds,
    })[["ID", "TargetRLF1", "TargetR1F1", "TargetLLM"]]

    # Sanity checks
    assert list(sub.columns) == ["ID", "TargetRLF1", "TargetR1F1", "TargetLLM"]
    assert len(sub) == len(test), f"Row count mismatch: {len(sub)} vs {len(test)}"
    assert sub.notna().all().all(), "Missing values in submission"
    assert (sub["TargetRLF1"] == sub["TargetR1F1"]).all()
    assert (sub["TargetRLF1"] == sub["TargetLLM"]).all()

    sub.to_csv(output_path, index=False, encoding="utf-8")
    tag = f" [{label}]" if label else ""
    print(f" Submission saved{tag}: {output_path}  ({len(sub):,} rows)")
    print("   ← Upload THIS CSV file to Zindi, not the notebook")
    return sub

print(" make_submission() ready")

 make_submission() ready


## 8 — Baseline 1: TF-IDF Retrieval

Finds the most similar training question per language using character n-gram TF-IDF,
then returns that training answer. Runs in ~2 minutes on CPU.

**Submit `submission_tfidf_baseline.csv` to get your first leaderboard anchor score.**

In [49]:
class TfidfRetrievalAnswerer:
    """Per-language TF-IDF nearest-neighbour retrieval."""

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col   = answer_col
        self.group_col    = group_col
        self.ngram_range  = ngram_range
        self.max_features = max_features
        self.models       = {}
        self.global_model = None

    def _fit_single(self, df):
        questions = df[self.question_col].fillna("").astype(str).tolist()
        answers   = df[self.answer_col].fillna("").astype(str).tolist()
        vec = TfidfVectorizer(
            analyzer="char_wb", ngram_range=self.ngram_range,
            min_df=1, max_features=self.max_features, lowercase=False,
        )
        X  = vec.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(X)
        return {"vec": vec, "nn": nn,
                "answers": np.array(answers, dtype=object),
                "questions": np.array(questions, dtype=object)}

    def fit(self, df):
        self.global_model = self._fit_single(df)
        if self.group_col and self.group_col in df.columns:
            for grp, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[grp] = self._fit_single(sub)
        print(f"  Fitted global model + {len(self.models)} per-language models")
        return self

    def predict_one(self, question, group=None):
        m = self.models.get(group, self.global_model) if group else self.global_model
        Xq = m["vec"].transform([question])
        dist, idx = m["nn"].kneighbors(Xq)
        i = idx[0][0]
        return m["answers"][i], 1 - float(dist[0][0]), m["questions"][i]

    def predict(self, df, question_col, group_col=None):
        preds, sims, matched = [], [], []
        for _, row in df.iterrows():
            q   = clean_text(row[question_col])
            grp = row[group_col] if group_col and group_col in df.columns else None
            ans, sim, mq = self.predict_one(q, grp)
            preds.append(ans); sims.append(sim); matched.append(mq)
        return preds, sims, matched

print(" TfidfRetrievalAnswerer defined")

 TfidfRetrievalAnswerer defined


In [ ]:
print("Training TF-IDF on train set...")
tfidf_model = TfidfRetrievalAnswerer(
    question_col=QUESTION_COL, answer_col=ANSWER_COL, group_col=LANG_COL
).fit(train)

print("Running validation predictions...")
val_preds_tfidf, val_sims, _ = tfidf_model.predict(val, QUESTION_COL, LANG_COL)

metrics_tfidf = compute_rouge(val_preds_tfidf, val[ANSWER_COL].tolist())
print(f"\n TF-IDF Baseline — Val ROUGE")
print(f"   ROUGE-1 F1 : {metrics_tfidf['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1 : {metrics_tfidf['rougeL_f1']:.4f}")

print("\nPer-language breakdown:")
lang_df = compute_rouge_by_language(val_preds_tfidf, val[ANSWER_COL].tolist(), val[LANG_COL].tolist())
print(lang_df.round(4).to_string())

Training TF-IDF on train set...
  Fitted global model + 8 per-language models
Running validation predictions...


In [ ]:
# generate test predictions and save submission
print("Generating TF-IDF test predictions...")
test_preds_tfidf, _, _ = tfidf_model.predict(test, QUESTION_COL, LANG_COL)

sub_tfidf = make_submission(
    ids=test[ID_COL].values,
    predictions=test_preds_tfidf,
    output_path=OUTPUT_TFIDF,
    label="Baseline 1 TF-IDF"
)
sub_tfidf.head(3)

Generating TF-IDF test predictions...
 Submission saved [Baseline 1 TF-IDF]: outputs\submission_tfidf_baseline.csv  (2,618 rows)
   ← Upload THIS CSV file to Zindi, not the notebook


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n..."
1,ID_TS_Aka_Gha_1C80317F,"Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ..."
2,ID_TS_Aka_Gha_06671AD1,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...


## 9 — Baseline 2: Zero-Shot LLM

Load a pretrained multilingual model and generate answers without any fine-tuning.
Use this as second Zindi submission.

**Recommended model to start:** `google/mt5-small` (fastest, downloads ~1.2 GB)
Change to `facebook/nllb-200-distilled-600M` for better African language coverage.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Config — change MODEL_NAME to experiment
MODEL_NAME        = "google/mt5-small"   # swap: mt5-base | nllb-200-distilled-600M
MAX_INPUT_LENGTH  = 256
MAX_OUTPUT_LENGTH = 512
BATCH_SIZE_LLM    = 4    # lower to 4 if OOM errors
NUM_BEAMS         = 4

# DEVICE is defined in Cell 4 and is always available regardless of run order
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cpu
Model  : google/mt5-small


In [ ]:
print(f"Loading {MODEL_NAME} (first run downloads weights)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model_llm = model_llm.to(DEVICE)
model_llm.eval()
print(f" Loaded — {sum(p.numel() for p in model_llm.parameters())/1e6:.0f}M params")

Loading google/mt5-small (first run downloads weights)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 192/192 [00:00<00:00, 7203.29it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


 Loaded — 300M params


In [ ]:
def build_prompt(question: str, language: str = None) -> str:
    """
    Input prompt fed to the model encoder.
    Experiment idea: try with/without the language tag and compare ROUGE scores.
    """
    q = str(question).strip()
    if language:
        lang_name = subset_to_language_name(language)
        return f"Answer this health question in {lang_name}: {q}"
    return f"Answer this health question: {q}"

def generate_answers_batch(questions, languages=None, batch_size=BATCH_SIZE_LLM):
    """Batched inference with beam search. Returns list of answer strings."""
    if languages is None:
        languages = [None] * len(questions)
    all_answers = []
    n = len(questions)
    for start in range(0, n, batch_size):
        end     = min(start + batch_size, n)
        prompts = [build_prompt(q, l) for q, l in
                   zip(questions[start:end], languages[start:end])]
        inputs  = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_INPUT_LENGTH
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model_llm.generate(
                **inputs,
                max_new_tokens=MAX_OUTPUT_LENGTH,
                num_beams=NUM_BEAMS,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = [re.sub(r"<extra_id_\d+>", "", a).strip() for a in decoded]
        all_answers.extend(cleaned)
        print(f"  {end}/{n} done", end="\r")
    print()
    return all_answers

print("generate_answers_batch() ready")

generate_answers_batch() ready


In [ ]:
# sanity check on 3 val examples
sample3 = val.head(3)
qs = sample3[QUESTION_COL].tolist()
ls = sample3[LANG_COL].tolist()
rs = sample3[ANSWER_COL].tolist()
gs = generate_answers_batch(qs, ls, batch_size=3)
for i, (q, ref, gen) in enumerate(zip(qs, rs, gs)):
    print(f"[{i+1}] {ls[i]}")
    print(f"  Q  : {q[:100]}")
    print(f"  Ref: {ref[:100]}")
    print(f"  Gen: {gen[:100]}")
    print()

  3/3 done
[1] Aka_Gha
  Q  : Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, at
  Ref: Nhyehyɛeɛ aa ama ne mu so te sɛ senea aborɔfo ka no 'STEM' ne 'vocational training' se ɛbɛ adrɛse mm
  Gen: .

[2] Aka_Gha
  Q  : Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?
  Ref: Nna ne awo hokwan ahorow a wɔte ase no ma mmabun tumi: Si gyinae a ɛfata wɔ wɔn nipadua, nna, ne abu
  Gen: 

[3] Aka_Gha
  Q  : Mɛyɛ dɛn atumi abɔ asisifo ho amanneɛ wɔ ɔkwan a etu mpɔn na ahobammɔ wom so, na anammɔn bɛn na metu
  Ref: Ayayade ho nsɛm a wɔbɛbɔ ho amanneɛ yiye na ahobammɔ wom no hwehwɛ sɛ wɔyɛ nneɛma a edidi so yi: Wɔk
  Gen: .



In [ ]:
# Validate zero-shot LLM on the Val set
# dont forget that the 200 samples is fine for quick iteration. so before logging Experiment 2
# toreport, i have to  re-run with VALIDATION_SAMPLE_SIZE = None for the full score.
VALIDATION_SAMPLE_SIZE = None  # set to None for full val (6k rows, which will be slower)

val_sample = val.sample(n=min(VALIDATION_SAMPLE_SIZE or len(val), len(val)), random_state=SEED)
val_qs     = val_sample[QUESTION_COL].tolist()
val_ls     = val_sample[LANG_COL].tolist()
val_refs   = val_sample[ANSWER_COL].tolist()

print(f"Evaluating zero-shot LLM on {len(val_sample)} val examples...")
val_preds_llm = generate_answers_batch(val_qs, val_ls)

metrics_llm = compute_rouge(val_preds_llm, val_refs)
print(f"\n Zero-Shot LLM ({MODEL_NAME}) — Val ROUGE (n={len(val_sample)})")
print(f"   ROUGE-1 F1 : {metrics_llm['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1 : {metrics_llm['rougeL_f1']:.4f}")
if VALIDATION_SAMPLE_SIZE:
    print(f"\n  Sampled result done.")

Evaluating zero-shot LLM on 6686 val examples...
  6686/6686 done

 Zero-Shot LLM (google/mt5-small) — Val ROUGE (n=6686)
   ROUGE-1 F1 : 0.0007
   ROUGE-L F1 : 0.0006


In [ ]:
# generate and save zero-shot submission
print(f"Generating test predictions ({len(test):,} rows)...")
test_preds_llm = generate_answers_batch(
    test[QUESTION_COL].tolist(), test[LANG_COL].tolist()
)
sub_llm = make_submission(
    ids=test[ID_COL].values,
    predictions=test_preds_llm,
    output_path=OUTPUT_LLM,
    label=f"Baseline 2 Zero-shot {MODEL_NAME}"
)
sub_llm.head(3)

Generating test predictions (2,618 rows)...
  2618/2618 done
 Submission saved [Baseline 2 Zero-shot google/mt5-small]: outputs\submission_llm_zeroshot.csv  (2,618 rows)
   ← Upload THIS CSV file to Zindi, not the notebook


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,nneɛma,nneɛma,nneɛma
1,ID_TS_Aka_Gha_1C80317F,?eловна,?eловна,?eловна
2,ID_TS_Aka_Gha_06671AD1,.,.,.


In [ ]:
# Check what's missing
sample = pd.read_csv(SAMPLE_SUB_PATH)
your_sub = pd.read_csv(OUTPUT_TFIDF)  # or whichever file you just submitted

sample_ids = set(sample['ID'])
your_ids   = set(your_sub['ID'])

missing = sample_ids - your_ids
extra   = your_ids - sample_ids

print(f"Zindi expects : {len(sample_ids):,} rows")
print(f"Your file has : {len(your_ids):,} rows")
print(f"Missing IDs   : {len(missing)}")
print(f"Extra IDs     : {len(extra)}")
print(f"\nSample missing: {list(missing)[:5]}")

Zindi expects : 2,618 rows
Your file has : 2,618 rows
Missing IDs   : 0
Extra IDs     : 0

Sample missing: []


## 10 — Compare Baselines

In [ ]:
# Hardcode TF-IDF results from Experiment 1 (already run and submitted)
metrics_tfidf = {"rouge1_f1": 0.4207, "rougeL_f1": 0.3654}

comparison = pd.DataFrame({
    "Baseline":   ["TF-IDF Retrieval", f"Zero-shot LLM ({MODEL_NAME})"],
    "ROUGE-1 F1": [metrics_tfidf["rouge1_f1"], metrics_llm["rouge1_f1"]],
    "ROUGE-L F1": [metrics_tfidf["rougeL_f1"], metrics_llm["rougeL_f1"]],
})
print("Baseline Comparison (validation set):")
print(comparison.round(4).to_string(index=False))
print()
print("Note: zero-shot LLM hasn't been trained on this data yet.")
print("Fine-tuning (next section) will significantly improve scores.")

Baseline Comparison (validation set):
                        Baseline  ROUGE-1 F1  ROUGE-L F1
                TF-IDF Retrieval      0.4207      0.3654
Zero-shot LLM (google/mt5-small)      0.0007      0.0006

Note: zero-shot LLM hasn't been trained on this data yet.
Fine-tuning (next section) will significantly improve scores.


## 11 — Baseline 3: Fine-tune the LLM

Fine-tuning adapts the model weights to the health QA task and all five languages.
This is where real performance gains come from.

**This is also the starting point for your 10 experiments** — change one thing at a time:
model size, learning rate, epochs, prompt strategy, batch size, beam width, etc.

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
)
from datasets import Dataset

#fine-tuning config — this is your Experiment 3 knob-board
EXPERIMENT_NAME        = "exp03_mt5small_finetuned"   # update per experiment
FINETUNE_OUTPUT_DIR    = str(CHECKPOINT_DIR / 'checkpoints' / EXPERIMENT_NAME)
FINETUNE_EPOCHS        = 4
FINETUNE_BATCH_SIZE    = 4    # lower to 4 if OOM
FINETUNE_LR            = 5e-5
FINETUNE_MAX_INPUT     = 128
FINETUNE_MAX_TARGET    = 256
FINETUNE_GRAD_ACCUM    = 2       # effective batch = BATCH * GRAD_ACCUM
FINETUNE_WARMUP_RATIO  = 0.1
OUTPUT_FINETUNED       = OUTPUT_DIR / f"submission_{EXPERIMENT_NAME}.csv"

import os
os.makedirs(FINETUNE_OUTPUT_DIR, exist_ok=True)
print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Model      : {MODEL_NAME}")
print(f"Epochs     : {FINETUNE_EPOCHS}")
print(f"LR         : {FINETUNE_LR}")
print(f"Batch size : {FINETUNE_BATCH_SIZE} × {FINETUNE_GRAD_ACCUM} grad accum steps")

Experiment : exp03_mt5small_finetuned
Model      : google/mt5-small
Epochs     : 4
LR         : 5e-05
Batch size : 4 × 2 grad accum steps


In [ ]:
def make_hf_dataset(df, question_col, answer_col, lang_col):
    """Convert DataFrame to HuggingFace Dataset with tokenised inputs & labels."""
    records = [
        {
            "prompt": build_prompt(str(row[question_col]),
                                   str(row[lang_col]) if lang_col in df.columns else None),
            "answer": str(row[answer_col])
        }
        for _, row in df.iterrows()
    ]
    raw_ds = Dataset.from_list(records)

    def preprocess(examples):
        model_inputs = tokenizer(
            examples["prompt"], max_length=FINETUNE_MAX_INPUT,
            truncation=True, padding=False
        )
        labels = tokenizer(
            text_target=examples["answer"], max_length=FINETUNE_MAX_TARGET,
            truncation=True, padding=False
        )
        # Mask padding tokens so loss ignores them
        model_inputs["labels"] = [
            [(t if t != tokenizer.pad_token_id else -100) for t in seq]
            for seq in labels["input_ids"]
        ]
        return model_inputs

    return raw_ds.map(preprocess, batched=True, remove_columns=["prompt", "answer"])

# use the provided Val.csv as validation (don't split train)
print("Building HuggingFace datasets...")
hf_train = make_hf_dataset(train, QUESTION_COL, ANSWER_COL, LANG_COL)
hf_val   = make_hf_dataset(val,   QUESTION_COL, ANSWER_COL, LANG_COL)
print(f"  Train: {len(hf_train):,} examples")
print(f"  Val  : {len(hf_val):,} examples")

Building HuggingFace datasets...


Map: 100%|██████████| 6686/6686 [00:06<00:00, 993.73 examples/s] 

  Train: 29,814 examples
  Val  : 6,686 examples


In [ ]:
from transformers import DataCollatorForSeq2Seq
print('imported')

imported


In [ ]:
import gc
import torch

# Clear everything from GPU memory
if 'model_llm' in dir():
    del model_llm
if 'trainer' in dir():
    del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Reload model with float16 to save VRAM
print("Reloading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model_llm = model_llm.to(DEVICE)
model_llm.train()

#free, total = torch.cuda.mem_get_info()
#print(f"Model loaded | GPU free: {free/1e9:.1f} GB / {total/1e9:.1f} GB")

Reloading model...


Loading weights: 100%|██████████| 192/192 [00:01<00:00, 117.10it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [ ]:
import numpy as _np
from rouge_score import rouge_scorer as _rs

#ROUGE metric for trainer (logs per epoch for the learning curve)
def compute_metrics_trainer(eval_pred):
    preds, labels = eval_pred
    # preds may be logits — decode token ids
    if hasattr(preds, "__len__") and len(preds) > 0 and not isinstance(preds[0], int):
        # already token ids from predict_with_generate
        pass
    labels = _np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    scorer_ = _rs.RougeScorer(["rouge1","rougeL"], tokenizer=WhitespaceTokenizer(), use_stemmer=False)
    r1, rl = [], []
    for p, r in zip(decoded_preds, decoded_labels):
        s = scorer_.score(r.strip(), p.strip())
        r1.append(s["rouge1"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
    return {"rouge1": float(_np.mean(r1)), "rougeL": float(_np.mean(rl))}

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model_llm,
    label_pad_token_id=-100, pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir=FINETUNE_OUTPUT_DIR,
    num_train_epochs=FINETUNE_EPOCHS,
    per_device_train_batch_size=FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size=FINETUNE_BATCH_SIZE,
    gradient_accumulation_steps=FINETUNE_GRAD_ACCUM,
    learning_rate=FINETUNE_LR,
    warmup_ratio=FINETUNE_WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    predict_with_generate=True,
    generation_max_length=FINETUNE_MAX_TARGET,
    bf16=(DEVICE == "cuda" and torch.cuda.is_bf16_supported()),
    fp16=(DEVICE == "cuda" and not torch.cuda.is_bf16_supported()),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",   # now optimise for ROUGE-1, not loss
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model_llm,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_trainer,
)

print(f"Starting fine-tuning — {EXPERIMENT_NAME}")
trainer.train()
print("Fine-tuning complete")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning — exp03_mt5small_finetuned


In [ ]:
import torch
print(torch.cuda.is_available())        # must print True
print(torch.cuda.get_device_name(0))    # must print something like Tesla T4

True
Tesla T4


In [ ]:
# Plot training curves — loss and ROUGE per epoch — save for report
logs = trainer.state.log_history

train_loss  = [(l["epoch"], l["loss"])       for l in logs if "loss" in l and "eval_loss" not in l]
eval_loss   = [(l["epoch"], l["eval_loss"])  for l in logs if "eval_loss" in l]
eval_rouge1 = [(l["epoch"], l["eval_rouge1"]) for l in logs if "eval_rouge1" in l]
eval_rougeL = [(l["epoch"], l["eval_rougeL"]) for l in logs if "eval_rougeL" in l]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: loss curves
if train_loss:
    te, tl = zip(*train_loss)
    axes[0].plot(te, tl, "b-o", markersize=3, label="Train loss")
if eval_loss:
    ee, el = zip(*eval_loss)
    axes[0].plot(ee, el, "r-s", markersize=5, label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title(f"Loss — {EXPERIMENT_NAME}")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: ROUGE curves
if eval_rouge1:
    er1e, er1v = zip(*eval_rouge1)
    axes[1].plot(er1e, er1v, "g-o", markersize=5, label="ROUGE-1")
if eval_rougeL:
    erLe, erLv = zip(*eval_rougeL)
    axes[1].plot(erLe, erLv, "m-s", markersize=5, label="ROUGE-L")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1 Score")
axes[1].set_title(f"Val ROUGE — {EXPERIMENT_NAME}")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(EXPERIMENT_NAME, fontsize=11, fontweight="bold")
plt.tight_layout()
curve_path = OUTPUT_DIR / f"training_curves_{EXPERIMENT_NAME}.png"
plt.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {curve_path}")

In [ ]:
# Evaluate fine-tuned model on full Val set
# model_llm holds the best checkpoint only if trainer.train() ran this session.
# after a Colab reset, reload: model_llm = AutoModelForSeq2SeqLM.from_pretrained(FINETUNE_OUTPUT_DIR).to(DEVICE)

print("Evaluating fine-tuned model on full Val set...")
model_llm.eval()
val_preds_ft = generate_answers_batch(val[QUESTION_COL].tolist(), val[LANG_COL].tolist())
metrics_ft   = compute_rouge(val_preds_ft, val[ANSWER_COL].tolist())
print(f"\n Fine-tuned ({EXPERIMENT_NAME}) — Val ROUGE")
print(f"   ROUGE-1 F1 : {metrics_ft['rouge1_f1']:.4f}")
print(f"   ROUGE-L F1 : {metrics_ft['rougeL_f1']:.4f}")

#per-language ROUGE table
print("\nPer-language breakdown:")
lang_df_ft = compute_rouge_by_language(
    val_preds_ft, val[ANSWER_COL].tolist(), val[LANG_COL].tolist()
)
print(lang_df_ft.round(4).to_string())

#per-language bar chart (save for report)
langs  = lang_df_ft.index.tolist()
r1vals = lang_df_ft["rouge1_f1"].tolist()
rLvals = lang_df_ft["rougeL_f1"].tolist()
x      = range(len(langs))
w      = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w/2 for i in x], r1vals, w, label="ROUGE-1", color="steelblue")
ax.bar([i + w/2 for i in x], rLvals, w, label="ROUGE-L", color="coral")
ax.set_xticks(list(x))
ax.set_xticklabels([l.replace("_", "\n") for l in langs], fontsize=10)
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1)
ax.set_title(f"Per-Language ROUGE — {EXPERIMENT_NAME}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
chart_path = OUTPUT_DIR / f"per_lang_rouge_{EXPERIMENT_NAME}.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart_path}")

In [ ]:
# generate test predictions and save submission
print(f"Generating test predictions for {len(test):,} rows...")
test_preds_ft = generate_answers_batch(test[QUESTION_COL].tolist(), test[LANG_COL].tolist())

sub_ft = make_submission(
    ids=test[ID_COL].values,
    predictions=test_preds_ft,
    output_path=OUTPUT_FINETUNED,
    label=EXPERIMENT_NAME
)
sub_ft.head(3)

## 12 — Experiment Log

Run this after every experiment to keep a clean record for your report.

In [ ]:
# fill in leaderboard_score after submitting to Zindi
experiment_entry = {
    "experiment":        EXPERIMENT_NAME,
    "model":             MODEL_NAME,
    "epochs":            FINETUNE_EPOCHS,
    "batch_size":        FINETUNE_BATCH_SIZE,
    "grad_accum":        FINETUNE_GRAD_ACCUM,
    "learning_rate":     FINETUNE_LR,
    "max_input_len":     FINETUNE_MAX_INPUT,
    "max_target_len":    FINETUNE_MAX_TARGET,
    "prompt_strategy":   "with_language_tag",
    "num_beams":         NUM_BEAMS,
    "val_rouge1":        round(metrics_ft["rouge1_f1"], 4),
    "val_rougeL":        round(metrics_ft["rougeL_f1"], 4),
    "leaderboard_score": "TBD",   # <-- paste your Zindi score here after submitting
    "notes":             "Full fine-tune, mt5-small, cosine LR, with language tag in prompt"
}

log_path = OUTPUT_DIR / "experiment_log.json"
# Load existing log or start fresh
if log_path.exists():
    with open(log_path) as f:
        log = json.load(f)
else:
    log = []

log.append(experiment_entry)
with open(log_path, "w") as f:
    json.dump(log, f, indent=2)

print(f" Experiment logged to {log_path}")
print(json.dumps(experiment_entry, indent=2))

## 13 — Your 10 Experiments Roadmap

Change **one variable at a time**, log results, repeat.

| # | `EXPERIMENT_NAME` | What to change | Insight you'll gain |
|---|---|---|---|
| 1 | `exp01_tfidf` | TF-IDF baseline | Anchor score |
| 2 | `exp02_zeroshot_mt5small` | Zero-shot mt5-small | Value of fine-tuning |
| 3 | `exp03_mt5small_finetuned` | Fine-tune mt5-small | ← **You are here** |
| 4 | `exp04_mt5base` | Swap to `mt5-base` | Model size effect |
| 5 | `exp05_nllb200` | Swap to `nllb-200-distilled-600M` | African lang coverage |
| 6 | `exp06_no_lang_tag` | Remove language from prompt | Prompt format effect |
| 7 | `exp07_more_epochs` | `FINETUNE_EPOCHS = 6` | Overfitting vs gains |
| 8 | `exp08_lower_lr` | `FINETUNE_LR = 1e-5` | LR sensitivity |
| 9 | `exp09_more_beams` | `NUM_BEAMS = 8` at inference | Decoding quality |
| 10 | `exp10_length_penalty` | Add `length_penalty=1.5` to `.generate()` | Answer completeness |

**Workflow per experiment:**
1. Update `EXPERIMENT_NAME` and the one variable you're changing
2. Re-run cells 11 onward
3. Check val ROUGE, submit to Zindi, paste leaderboard score into Cell 12
4. Git commit: `git commit -am "exp04: mt5-base — val rouge1=0.XX"`

In [ ]:
import pandas as pd
from pathlib import Path

# Read files completely fresh, bypassing any earlier processing
sample_raw = pd.read_csv(SAMPLE_SUB_PATH)
test_raw   = pd.read_csv(TEST_PATH)

print(f"Sample submission expects : {len(sample_raw)} rows")
print(f"Test.csv raw rows         : {len(test_raw)} rows")
print(f"Current test variable     : {len(test)} rows")
print()

# Check which IDs are missing
sample_ids = set(sample_raw['ID'])
test_ids   = set(test_raw['ID'])
current_ids = set(test['ID'])

print(f"IDs in sample but not in Test.csv       : {len(sample_ids - test_ids)}")
print(f"IDs in sample but not in current test   : {len(sample_ids - current_ids)}")
print(f"IDs in Test.csv but not in sample       : {len(test_ids - sample_ids)}")
print()
print("Sample ID prefix examples:")
print(list(sample_ids)[:5])
print("Test.csv ID prefix examples:")
print(list(test_ids)[:5])

Sample submission expects : 2618 rows
Test.csv raw rows         : 2618 rows
Current test variable     : 2618 rows

IDs in sample but not in Test.csv       : 0
IDs in sample but not in current test   : 0
IDs in Test.csv but not in sample       : 0

Sample ID prefix examples:
['ID_TS_Eng_Eth_550A1B46', 'ID_TS_Swa_Ken_CF1D7D3A', 'ID_TS_Aka_Gha_50C187FC', 'ID_TS_Aka_Gha_508EBAD7', 'ID_TS_Eng_Uga_DD30DE89']
Test.csv ID prefix examples:
['ID_TS_Eng_Eth_550A1B46', 'ID_TS_Swa_Ken_CF1D7D3A', 'ID_TS_Aka_Gha_50C187FC', 'ID_TS_Aka_Gha_508EBAD7', 'ID_TS_Eng_Uga_DD30DE89']


In [ ]:
# verify what's actually in your saved submission file
import pandas as pd
old_sub = pd.read_csv('./outputs/submission_llm_zeroshot.csv')
print(f"Rows in saved CSV : {len(old_sub)}")
print(f"Rows expected     : 2618")

Rows in saved CSV : 2618
Rows expected     : 2618


In [ ]:
old_sub = pd.read_csv('./outputs/submission_llm_zeroshot.csv')
print("Columns:", old_sub.columns.tolist())
print("Shape:", old_sub.shape)
print()

# Check for any null or empty values
print("Null counts:")
print(old_sub.isnull().sum())
print()

# Check ID format matches sample
sample_raw = pd.read_csv(SAMPLE_SUB_PATH)
print("Sample columns:", sample_raw.columns.tolist())
print()

# Compare a few IDs
print("Your IDs (first 3):")
print(old_sub['ID'].head(3).tolist())
print()
print("Sample IDs (first 3):")
print(sample_raw['ID'].head(3).tolist())
print()

# Check if IDs match exactly
missing = set(sample_raw['ID']) - set(old_sub['ID'])
print(f"IDs in sample but missing from your file: {len(missing)}")
if missing:
    print("Missing examples:", list(missing)[:5])

Columns: ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
Shape: (2618, 4)

Null counts:
ID            0
TargetRLF1    0
TargetR1F1    0
TargetLLM     0
dtype: int64

Sample columns: ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']

Your IDs (first 3):
['ID_TS_Aka_Gha_A3B1799D', 'ID_TS_Aka_Gha_1C80317F', 'ID_TS_Aka_Gha_06671AD1']

Sample IDs (first 3):
['ID_TS_Aka_Gha_A3B1799D', 'ID_TS_Aka_Gha_1C80317F', 'ID_TS_Aka_Gha_06671AD1']

IDs in sample but missing from your file: 0


In [ ]:
print("Regenerating predictions...")
test_preds_tfidf, _, _ = tfidf_model.predict(test, QUESTION_COL, LANG_COL)

sub_llm = make_submission(
    ids=test[ID_COL].values,
    predictions=test_preds_tfidf,
    output_path='./outputs/submission_llm_zeroshot.csv',
    label="submission_llm_zeroshot — fixed"
)
print(f"New submission rows: {len(sub_tfidf)}")
sub_tfidf.head(3)

Regenerating predictions...
 Submission saved [submission_llm_zeroshot — fixed]: ./outputs/submission_llm_zeroshot.csv  (2,618 rows)
   ← Upload THIS CSV file to Zindi, not the notebook
New submission rows: 2618


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n..."
1,ID_TS_Aka_Gha_1C80317F,"Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ..."
2,ID_TS_Aka_Gha_06671AD1,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...
